Telemetry Anomaly Detection

## Dataset (Reference link)

We use NASA SMAP spacecraft telemetry data with expert-labeled anomaly intervals.
Link to dataset:- https://www.kaggle.com/datasets/patrickfleith/nasa-anomaly-detection-dataset-smap-msl

## Importing Libraries

In [ ]:
!pip install xgboost -q
from xgboost import XGBClassifier

## Loading Data

In [ ]:
base_path = '/kaggle/input/nasa-anomaly-detection-dataset-smap-msl'
train_path = f'{base_path}/data/data/train'
test_path = f'{base_path}/data/data/test'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import ast
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, r2_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

import warnings
warnings.filterwarnings('ignore')

In [ ]:
anomaly_df = pd.read_csv(f'{base_path}/labeled_anomalies.csv')
anomaly_df.head()

In [ ]:
print(f"Total labeled channels: {len(anomaly_df)}")
anomaly_df.info()

In [ ]:
anomaly_df['anomaly_sequences'] = anomaly_df['anomaly_sequences'].apply(ast.literal_eval)
anomaly_df['num_anomaly_ranges'] = anomaly_df['anomaly_sequences'].apply(len)

In [ ]:
print("Anomaly distribution by spacecraft:")
anomaly_df.groupby('spacecraft').agg({'chan_id': 'count', 'num_values': 'sum'}).rename(columns={'chan_id': 'channels', 'num_values': 'total_values'})

In [ ]:
train_files = os.listdir(train_path)
test_files = os.listdir(test_path)

print(f"Training files: {len(train_files)}")
print(f"Testing files: {len(test_files)}")

## Exploratory Data Analysis

In [ ]:
sample_channel = train_files[0]
sample_data = np.load(f'{train_path}/{sample_channel}')

print(f"Sample channel: {sample_channel}")
print(f"Shape: {sample_data.shape}")
print(f"Data type: {sample_data.dtype}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6))

for idx, ax in enumerate(axes.flat):
    if idx < len(train_files):
        channel_name = train_files[idx]
        data = np.load(f'{train_path}/{channel_name}')
        ax.plot(data[:, 0], linewidth=0.5)
        ax.set_title(channel_name.replace('.npy', ''))
        ax.set_xlabel('Time Step')
        ax.set_ylabel('Value')

plt.tight_layout()
plt.show()

### Anomalous Region Visualization

In [ ]:
anomaly_channel = anomaly_df.iloc[0]
channel_id = anomaly_channel['chan_id']
anomaly_ranges = anomaly_channel['anomaly_sequences']

print(f"Channel: {channel_id}")
print(f"Spacecraft: {anomaly_channel['spacecraft']}")
print(f"Anomaly ranges: {anomaly_ranges}")

In [ ]:
test_data = np.load(f'{test_path}/{channel_id}.npy')

plt.figure(figsize=(14, 4))
plt.plot(test_data[:, 0], linewidth=0.5, label='Telemetry Signal')

for start, end in anomaly_ranges:
    plt.axvspan(start, end, alpha=0.3, color='red', label='Anomaly')

plt.title(f'Test Data - {channel_id}')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.tight_layout()
plt.show()

## Feature Engineering

Statistical features extracted from sliding windows of telemetry signals.

In [ ]:
def extract_features(signal, window_size=50, step=25):
    features_list = []
    
    for i in range(0, len(signal) - window_size, step):
        window = signal[i:i + window_size]
        
        feat = {
            'mean': np.mean(window),
            'std': np.std(window),
            'min': np.min(window),
            'max': np.max(window),
            'range': np.max(window) - np.min(window),
            'skew': stats.skew(window.flatten()),
            'kurtosis': stats.kurtosis(window.flatten()),
            'energy': np.sum(window**2),
            'zero_crossings': np.sum(np.diff(np.sign(window.flatten() - np.mean(window))) != 0),
            'window_start': i,
            'window_end': i + window_size
        }
        
        features_list.append(feat)
    
    return pd.DataFrame(features_list)

In [ ]:
def create_labels(signal_length, anomaly_ranges, window_size=50, step=25):
    labels = []
    
    for i in range(0, signal_length - window_size, step):
        window_start = i
        window_end = i + window_size
        
        is_anomaly = 0
        for a_start, a_end in anomaly_ranges:
            if not (window_end < a_start or window_start > a_end):
                is_anomaly = 1
                break
        
        labels.append(is_anomaly)
    
    return np.array(labels)

### Building Dataset

In [ ]:
all_features = []
window_size = 50
step = 25

for idx, row in anomaly_df.iterrows():
    channel_id = row['chan_id']
    try:
        test_signal = np.load(f'{test_path}/{channel_id}.npy')
        
        if test_signal.ndim > 1:
            test_signal = test_signal[:, 0]
        
        features = extract_features(test_signal.reshape(-1, 1), window_size, step)
        
        anomaly_ranges = row['anomaly_sequences']
        labels = create_labels(len(test_signal), anomaly_ranges, window_size, step)
        
        features['channel_id'] = channel_id
        features['label'] = labels
        
        all_features.append(features)
        
    except Exception as e:
        print(f"Error processing {channel_id}: {e}")

dataset = pd.concat(all_features, ignore_index=True)
print(f"Total samples: {len(dataset)}")

In [ ]:
dataset.head()

In [ ]:
print("Class distribution:")
print(dataset['label'].value_counts())
print(f"Anomaly ratio: {dataset['label'].mean()*100:.2f}%")

## Data Preprocessing

In [ ]:
feature_cols = ['mean', 'std', 'min', 'max', 'range', 'skew', 'kurtosis', 'energy', 'zero_crossings']

X = dataset[feature_cols].copy()
y = dataset['label'].copy()

X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.median(), inplace=True)

print(f"Features: {X.shape}")
print(f"Labels: {y.shape}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)}")
print(f"Testing set: {len(X_test)}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Model Training

In [ ]:
def evaluate_model(model, X_tr, X_te, y_tr, y_te, name):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    
    y_prob = None
    roc_auc = None
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_te)[:, 1]
        roc_auc = roc_auc_score(y_te, y_prob)
    
    results = {
        'Model': name,
        'Accuracy': accuracy_score(y_te, y_pred),
        'Precision': precision_score(y_te, y_pred, zero_division=0),
        'Recall': recall_score(y_te, y_pred, zero_division=0),
        'F1-Score': f1_score(y_te, y_pred, zero_division=0),
        'ROC-AUC': roc_auc
    }
    
    return results, y_pred, y_prob

### Logistic Regression

In [ ]:
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_results, lr_pred, lr_prob = evaluate_model(lr_model, X_train_scaled, X_test_scaled, y_train, y_test, 'Logistic Regression')

for k, v in lr_results.items():
    if v is not None and k != 'Model':
        print(f"{k}: {v:.4f}")

### Random Forest

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_results, rf_pred, rf_prob = evaluate_model(rf_model, X_train_scaled, X_test_scaled, y_train, y_test, 'Random Forest')

for k, v in rf_results.items():
    if v is not None and k != 'Model':
        print(f"{k}: {v:.4f}")

### XGBoost

In [ ]:
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

xgb_model = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)

xgb_results, xgb_pred, xgb_prob = evaluate_model(xgb_model, X_train_scaled, X_test_scaled, y_train, y_test, 'XGBoost')

for k, v in xgb_results.items():
    if v is not None and k != 'Model':
        print(f"{k}: {v:.4f}")

### SVM

In [ ]:
svm_model = SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42)
svm_results, svm_pred, svm_prob = evaluate_model(svm_model, X_train_scaled, X_test_scaled, y_train, y_test, 'SVM')

for k, v in svm_results.items():
    if v is not None and k != 'Model':
        print(f"{k}: {v:.4f}")

## Results Comparison

In [ ]:
results_df = pd.DataFrame([lr_results, rf_results, xgb_results, svm_results])
results_df = results_df.set_index('Model').round(4)
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(results_df.index))
width = 0.2

for i, metric in enumerate(metrics):
    ax.bar(x + i*width, results_df[metric], width, label=metric)

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Comparison')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df.index, rotation=15)
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

### ROC Curves

In [ ]:
plt.figure(figsize=(8, 6))

model_probs = [
    ('Logistic Regression', lr_prob),
    ('Random Forest', rf_prob),
    ('XGBoost', xgb_prob),
    ('SVM', svm_prob)
]

for name, prob in model_probs:
    if prob is not None:
        fpr, tpr, _ = roc_curve(y_test, prob)
        auc = roc_auc_score(y_test, prob)
        plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.tight_layout()
plt.show()

### Confusion Matrix

In [ ]:
best_model = results_df['F1-Score'].idxmax()
print(f"Best model: {best_model}")

best_pred = {'Logistic Regression': lr_pred, 'Random Forest': rf_pred, 'XGBoost': xgb_pred, 'SVM': svm_pred}[best_model]

cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_model}')
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(y_test, best_pred, target_names=['Normal', 'Anomaly']))

## Feature Importance

In [ ]:
importance = pd.DataFrame({'Feature': feature_cols, 'Importance': rf_model.feature_importances_})
importance = importance.sort_values('Importance', ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(importance['Feature'], importance['Importance'])
plt.xlabel('Importance')
plt.title('Feature Importance (Random Forest)')
plt.tight_layout()
plt.show()

## Cross-Validation

In [ ]:
cv_f1 = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring='f1')
cv_acc = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring='accuracy')

print("5-Fold Cross-Validation (Random Forest):")
print(f"F1: {cv_f1.mean():.4f} (+/- {cv_f1.std()*2:.4f})")
print(f"Accuracy: {cv_acc.mean():.4f} (+/- {cv_acc.std()*2:.4f})")

## Summary

## R2-Score


In [ ]:
print("R² Scores (probability-based):")
for name, prob in model_probs:
    if prob is not None:
        r2 = r2_score(y_test, prob)
        print(f"{name}: {r2:.4f}")

In [ ]:
print("="*60)
print("FINAL RESULTS")
print("="*60)
print(f"Dataset: NASA SMAP & MSL Telemetry")
print(f"Total samples: {len(dataset)}")
print(f"Anomaly ratio: {dataset['label'].mean()*100:.2f}%")
print(f"Best model: {best_model}")
print("="*60)
print(results_df.to_string())
print("="*60)

In [ ]:
output_df = pd.DataFrame({
    "actual": y_test.values.flatten(),
    "predicted": np.asarray(best_pred).flatten()
})

output_df.to_csv("/kaggle/working/output.csv", index=False)
print("✅ output.csv saved!")